# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [3]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [4]:
# Task — Multi-series line with highlight
# Load data with a safe fallback in case the project path differs
import os
import pandas as pd
import plotly.express as px

data_path = "../data/co2_emissions.csv"
fallback_path = "/Users/tirumalaabhinavreddy/data-viz-class-material-w2026/data/co2_emissions.csv"
if not os.path.exists(data_path) and os.path.exists(fallback_path):
    data_path = fallback_path

df = pd.read_csv(data_path)
asia = df[df["Region"] == "Asia"].copy()
highlight = "China"

# All countries are visible for context; only one is highlighted in colour
color_map = {
    country: "#D9D9D9" if country != highlight else "#D62728"
    for country in asia["Country"].unique()
}

fig = px.line(
    asia,
    x="Year",
    y="CO2_Mt",
    color="Country",
    color_discrete_map=color_map,
    labels={"Year": "", "CO2_Mt": "CO2 emissions (Mt)"},
    template="plotly_white",
    height=500,
)

for trace in fig.data:
    trace.showlegend = False
    trace.line.width = 1.5
    if trace.name == highlight:
        trace.line.color = "#D62728"
        trace.line.width = 3
    else:
        trace.line.color = "#D9D9D9"

last_point = asia[asia["Country"] == highlight].sort_values("Year").iloc[-1]
fig.add_annotation(
    x=last_point["Year"],
    y=last_point["CO2_Mt"],
    text=highlight,
    showarrow=False,
    xanchor="left",
    xshift=8,
    font=dict(color="#D62728", size=12, family="Arial"),
)

fig.update_layout(
    title=(
        f"{highlight} drives the Asian emissions surge — a steep rise from 2000 to 2022"
    ),
    title_x=0.5,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(size=13, family="Arial"),
    xaxis=dict(showgrid=False, title=""),
    yaxis=dict(showgrid=True, gridcolor="#EEEEEE", title="CO2 emissions (Mt)"),
    margin=dict(l=60, r=80, t=80, b=50),
)

fig.show()
